In [ ]:
import os

In [1]:
import requests

r = requests.get(
    "https://api.elsevier.com/content/search/scopus",
    params={"query": 'SRCTITLE("American Journal of Neuroradiology")'},
    headers={"X-ELS-APIKey": "57581543dea4d979a748eb7941383971"}
)

print(r.status_code)
print(r.text[:500])

200
{"search-results":{"opensearch:totalResults":"16178","opensearch:startIndex":"0","opensearch:itemsPerPage":"25","opensearch:Query":{"@role": "request", "@searchTerms": "SRCTITLE(\"American Journal of Neuroradiology\")", "@startPage": "0"},"link": [{"@_fa": "true", "@ref": "self", "@href": "https://api.elsevier.com/content/search/scopus?start=0&count=25&query=SRCTITLE%28%22American+Journal+of+Neuroradiology%22%29", "@type": "application/json"},{"@_fa": "true", "@ref": "first", "@href": "https://a


In [2]:
import requests
import pandas as pd

API_KEY = "57581543dea4d979a748eb7941383971"

query = '''
SRCTITLE("American Journal of Neuroradiology")
AND TITLE-ABS-KEY(
    "artificial intelligence"
    OR "machine learning"
    OR "deep learning"
)
'''

url = "https://api.elsevier.com/content/search/scopus"

r = requests.get(
    url,
    params={
        "query": query,
        "count": 25
    },
    headers={
        "X-ELS-APIKey": API_KEY,
        "Accept": "application/json"
    }
)

data = r.json()

rows = []

for p in data["search-results"]["entry"]:
    rows.append({
        "title": p.get("dc:title"),
        "year": p.get("prism:coverDate", "")[:4],
        "citations": int(p.get("citedby-count", 0)),
        "authors": p.get("dc:creator"),
        "doi": p.get("prism:doi")
    })

df = pd.DataFrame(rows)
print(df.sort_values("citations", ascending=False))

                                                title  year  citations  \
24  Choroid Plexus Enlargement in Multiple Scleros...  2026          3   
18  Artificial Intelligence–Assisted Detection of ...  2026          2   
13  High-Resolution 3T MRI of the Membranous Labyr...  2026          1   
5   Predicting Intracranial Aneurysm Rupture Risk ...  2026          1   
2   Deep-Learning Accelerated Vessel Wall Imaging ...  2026          1   
19  Head-to-Head Comparison of 2 Artificial Intell...  2026          1   
0   Comparative Analysis of Artificial Intelligenc...  2026          0   
3   High-Resolution 2D versus 3D Lumbar Spine MRI ...  2026          0   
1   Deep Learning Radiomics Signature from Multico...  2026          0   
8   Multimodal CT Perfusion–Based Deep Learning fo...  2026          0   
7         Vessel Wall Imaging at 7T: State of the Art  2026          0   
6   Neuroimaging in Low- to Middle-Income Countrie...  2026          0   
4   Comparative Evaluation of Deep Lea

In [ ]:
API_KEY = "57581543dea4d979a748eb7941383971"


In [9]:
import time
import requests
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime


API_KEY = "57581543dea4d979a748eb7941383971"

BASE_URL = "https://api.elsevier.com/content/search/scopus"

today = datetime.today()
ten_years_ago = today.replace(year=today.year - 10)

START_YEAR = ten_years_ago.year
END_YEAR = today.year
CURRENT_YEAR = today.year

HEADERS = {
    "X-ELS-APIKey": API_KEY,
    "Accept": "application/json",
}

COUNT = 25

# Minimal query: fewer chances of Scopus syntax errors
QUERY = (
    f'ISSN(0195-6108) '
    f'AND PUBYEAR > {START_YEAR - 1} '
    f'AND PUBYEAR < {END_YEAR + 1}'
)

print("Query:")
print(QUERY)


def search_scopus(query, count=25):
    rows = []
    start = 0
    total_results = None

    while True:
        params = {
            "query": query,
            "count": count,
            "start": start,
        }

        response = requests.get(
            BASE_URL,
            headers=HEADERS,
            params=params,
            timeout=30,
        )

        if response.status_code != 200:
            print("Error status:", response.status_code)
            print("URL:", response.url)
            print(response.text[:2000])
            response.raise_for_status()

        data = response.json()
        results = data.get("search-results", {})

        if total_results is None:
            total_results = int(results.get("opensearch:totalResults", 0))
            print(f"Total AJNR results found: {total_results}")

            if total_results == 0:
                return pd.DataFrame()

        entries = results.get("entry", [])

        if not entries:
            break

        for paper in entries:
            rows.append({
                "title": paper.get("dc:title", ""),
                "year": paper.get("prism:coverDate", "")[:4],
                "cover_date": paper.get("prism:coverDate"),
                "journal": paper.get("prism:publicationName"),
                "authors": paper.get("dc:creator"),
                "doi": paper.get("prism:doi"),
                "eid": paper.get("eid"),
                "scopus_id": paper.get("dc:identifier"),
                "citedby_count": paper.get("citedby-count"),
                "subtype": paper.get("subtypeDescription"),
                "description": paper.get("dc:description", ""),
            })

        print(f"Retrieved {len(rows)} / {total_results}")

        start += count

        if start >= total_results:
            break

        time.sleep(0.2)

    return pd.DataFrame(rows)


df_all = search_scopus(QUERY, COUNT)

if df_all.empty:
    print("No AJNR papers found.")
    raise SystemExit


# Exact past-10-year filter
df_all["cover_date"] = pd.to_datetime(df_all["cover_date"], errors="coerce")

df_all = df_all[
    (df_all["cover_date"] >= ten_years_ago) &
    (df_all["cover_date"] <= today)
].copy()


# Local AI keyword filter
AI_KEYWORDS = [
    "artificial intelligence",
    "machine learning",
    "deep learning",
    "radiomics",
    "transformer",
    "chatgpt",
    "large language model",
    "foundation model",
    "neural network",
    "convolutional neural network",
    #"random forest",
    #"support vector machine",
    "cnn",
    "llm",
    #"computer-aided",
    #"computer aided",
    #"automated",
    #"automation",
    #"algorithm",
]


def is_ai_related(row):
    text = f"{row.get('title', '')} {row.get('description', '')}".lower()
    return any(keyword in text for keyword in AI_KEYWORDS)


df = df_all[df_all.apply(is_ai_related, axis=1)].copy()

if df.empty:
    print("No AI-related AJNR papers found after local keyword filtering.")
    df_all.to_csv("ajnr_all_past10years.csv", index=False)
    print("Saved all AJNR papers to ajnr_all_past10years.csv for inspection.")
    raise SystemExit


# Clean
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df["citedby_count"] = pd.to_numeric(df["citedby_count"], errors="coerce").fillna(0).astype(int)

df = df.dropna(subset=["year"])
df["year"] = df["year"].astype(int)

df["citations_per_year"] = df["citedby_count"] / (CURRENT_YEAR - df["year"] + 1)

df = df.sort_values(["year", "citedby_count"], ascending=[False, False])


# Save outputs
df_all.to_csv("ajnr_all_past10years.csv", index=False)
df.to_csv("ajnr_ai_scopus_past10years_from_today.csv", index=False)

print("\nSaved:")
print("ajnr_all_past10years.csv")
print("ajnr_ai_scopus_past10years_from_today.csv")


# Summaries
papers_per_year = (
    df.groupby("year")
    .size()
    .reset_index(name="n_papers")
    .sort_values("year")
)

citations_by_year = (
    df.groupby("year")["citedby_count"]
    .sum()
    .reset_index(name="total_citations")
    .sort_values("year")
)

top_cited = (
    df.sort_values("citedby_count", ascending=False)
    [["year", "cover_date", "citedby_count", "citations_per_year", "title", "authors", "doi"]]
    .head(25)
)

fastest_growing = (
    df.sort_values("citations_per_year", ascending=False)
    [["year", "cover_date", "citedby_count", "citations_per_year", "title", "authors", "doi"]]
    .head(25)
)

papers_per_year.to_csv("ajnr_ai_papers_per_year.csv", index=False)
citations_by_year.to_csv("ajnr_ai_citations_by_year.csv", index=False)
top_cited.to_csv("ajnr_ai_top25_most_cited.csv", index=False)
fastest_growing.to_csv("ajnr_ai_top25_fastest_growing.csv", index=False)

print("\nPapers per year:")
print(papers_per_year.to_string(index=False))

print("\nTop 25 most cited:")
print(top_cited.to_string(index=False))

print("\nTop 25 fastest growing:")
print(fastest_growing.to_string(index=False))


# Plots
plt.figure(figsize=(8, 5))
plt.bar(papers_per_year["year"], papers_per_year["n_papers"])
plt.xlabel("Publication Year")
plt.ylabel("Number of AJNR AI Papers")
plt.title("AJNR AI-related Papers per Year")
plt.xticks(papers_per_year["year"], rotation=45)
plt.tight_layout()
plt.savefig("ajnr_ai_papers_per_year.png", dpi=300)
plt.close()

plt.figure(figsize=(8, 5))
plt.bar(citations_by_year["year"], citations_by_year["total_citations"])
plt.xlabel("Publication Year")
plt.ylabel("Total Citations")
plt.title("Total Citations of AJNR AI-related Papers by Publication Year")
plt.xticks(citations_by_year["year"], rotation=45)
plt.tight_layout()
plt.savefig("ajnr_ai_citations_by_year.png", dpi=300)
plt.close()

print("\nDone.")

Query:
ISSN(0195-6108) AND PUBYEAR > 2015 AND PUBYEAR < 2027
Total AJNR results found: 3944
Retrieved 25 / 3944
Retrieved 50 / 3944
Retrieved 75 / 3944
Retrieved 100 / 3944
Retrieved 125 / 3944
Retrieved 150 / 3944
Retrieved 175 / 3944
Retrieved 200 / 3944
Retrieved 225 / 3944
Retrieved 250 / 3944
Retrieved 275 / 3944
Retrieved 300 / 3944
Retrieved 325 / 3944
Retrieved 350 / 3944
Retrieved 375 / 3944
Retrieved 400 / 3944
Retrieved 425 / 3944
Retrieved 450 / 3944
Retrieved 475 / 3944
Retrieved 500 / 3944
Retrieved 525 / 3944
Retrieved 550 / 3944
Retrieved 575 / 3944
Retrieved 600 / 3944
Retrieved 625 / 3944
Retrieved 650 / 3944
Retrieved 675 / 3944
Retrieved 700 / 3944
Retrieved 725 / 3944
Retrieved 750 / 3944
Retrieved 775 / 3944
Retrieved 800 / 3944
Retrieved 825 / 3944
Retrieved 850 / 3944
Retrieved 875 / 3944
Retrieved 900 / 3944
Retrieved 925 / 3944
Retrieved 950 / 3944
Retrieved 975 / 3944
Retrieved 1000 / 3944
Retrieved 1025 / 3944
Retrieved 1050 / 3944
Retrieved 1075 / 3944
Retr